# 🐋 Whale Bot ReDux — train on Colab (TPU or GPU)

Runs the full pipeline in Colab: **tagged hydrophone audio → mel-spectrograms → trained classifier**.

The training code auto-detects a TPU (via PyTorch/XLA), a GPU, or CPU, so the
only thing you change per runtime is the `USE_TPU` flag below.

> **TPU vs GPU:** for a ResNet18 on spectrograms, a **GPU (T4)** runtime is
> simpler and usually faster end-to-end than a TPU — XLA compilation overhead
> dominates at this model size. TPU pays off with much larger models/batches.
> Try both; flip `USE_TPU` and pick the runtime to match.

**Set the runtime first:** `Runtime → Change runtime type → TPU` (or `GPU`).

## 1. Configuration

Set `USE_TPU` to match the runtime you selected. `AUDIO_SOURCE` /
`ANNOTATIONS_SOURCE` point at folders in your Google Drive (set up in step 4).

In [ ]:
USE_TPU = True   # False -> use a GPU (or CPU) runtime instead

REPO_URL = "https://github.com/jking323/Whale-Bot-ReDux.git"
REPO_BRANCH = "claude/model-training-repo-setup-2nj7iz"  # change once merged to Alpha

# Where your recordings + tag files live in Google Drive (created/used in step 4).
DRIVE_ROOT = "/content/drive/MyDrive/whale_bot"
AUDIO_SOURCE = DRIVE_ROOT + "/audio"          # .wav / .flac / .mp3 recordings
ANNOTATIONS_SOURCE = DRIVE_ROOT + "/annotations"  # separate CSV / Audacity / Raven tags
MODEL_OUTPUT = DRIVE_ROOT + "/models"          # trained checkpoint copied back here

## 2. Clone the repository

In [ ]:
import os

if not os.path.isdir("/content/Whale-Bot-ReDux"):
    !git clone --branch {REPO_BRANCH} {REPO_URL} /content/Whale-Bot-ReDux
%cd /content/Whale-Bot-ReDux
!git log --oneline -1

## 3. Install dependencies

> ⚠️ **PyTorch/XLA is version-sensitive:** `torch`, `torchvision`, and
> `torch_xla` must share the same version. If the TPU install errors, check the
> current install line at <https://github.com/pytorch/xla> and adjust the
> versions below.

In [ ]:
# Audio stack (both runtimes)
!pip install -q librosa soundfile

if USE_TPU:
    # PyTorch/XLA for TPU — keep the three versions in lockstep.
    !pip install -q "torch~=2.5.0" "torchvision~=0.20.0" "torch_xla[tpu]~=2.5.0" \
        -f https://storage.googleapis.com/libtpu-releases/index.html
else:
    # GPU/CPU — torch + torchvision ship with Colab, but pin them if missing.
    !pip install -q torch torchvision

print("\nDependencies installed. If Colab asks to restart the runtime, do it, "
      "then re-run from step 2 (skip this cell).")

In [ ]:
# Sanity check: confirm the accelerator the trainer will pick up.
if USE_TPU:
    os.environ["PJRT_DEVICE"] = "TPU"  # inherited by the training subprocess
    import torch_xla.core.xla_model as xm
    print("XLA device:", xm.xla_device())
else:
    import torch
    print("CUDA available:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 4. Mount Google Drive

Drive holds your recordings, tags, and trained models so they survive Colab
sessions. On first run this creates `whale_bot/{audio,annotations,models}/` in
your Drive — upload recordings into `audio/` and tag files into `annotations/`.

**Tag format reminder** (see the repo README for Audacity/Raven variants) — a
CSV `annotations/tags.csv` like:

```
audio_file,start,end,label
rec_2024-01-05.wav,12.4,15.1,orca
rec_2024-01-05.wav,88.0,91.2,humpback
rec_2024-01-05.wav,140.0,143.0,noise
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

for path in (AUDIO_SOURCE, ANNOTATIONS_SOURCE, MODEL_OUTPUT):
    os.makedirs(path, exist_ok=True)

# Point the repo's data/ dirs at the Drive folders (no copying needed).
os.makedirs("/content/Whale-Bot-ReDux/data", exist_ok=True)
for name, target in (("audio", AUDIO_SOURCE), ("annotations", ANNOTATIONS_SOURCE)):
    link = f"/content/Whale-Bot-ReDux/data/{name}"
    if os.path.islink(link) or os.path.exists(link):
        !rm -rf {link}
    os.symlink(target, link)

print("audio files:", len(os.listdir(AUDIO_SOURCE)))
print("annotation files:", len(os.listdir(ANNOTATIONS_SOURCE)))

### Where to find hydrophone data

Open marine-mammal acoustic datasets to get started. Most ship with labels you
can convert into a `tags.csv` (`audio_file,start,end,label`):

| Source | What it is | Link |
|---|---|---|
| **Watkins Marine Mammal Sound Database** (WHOI) | Curated, labeled calls for ~60 species — a great first classifier target | https://cis.whoi.edu/science/B/whalesounds/ |
| **MobySound** | Research-grade, annotated marine mammal recordings from detection challenges | https://www.mobysound.org/ |
| **Orcasound** | Live + archived Salish Sea hydrophones (orca); open data mirrored on AWS | https://www.orcasound.net/data/ · https://registry.opendata.aws/orcasound/ |
| **NOAA Passive Acoustic Archive / SanctSound** | Long-term hydrophone recordings from US waters | https://www.ncei.noaa.gov/products/passive-acoustic-data |
| **DCLDE workshop datasets** | Benchmark detection/classification sets with annotations | https://www.soest.hawaii.edu/ore/dclde/ |
| **Kaggle — Whale Detection Challenge** | Right-whale up-call clips, pre-labeled | https://www.kaggle.com/c/whale-detection-challenge |

> Tags/labels come **separately** from the audio here: download recordings into
> `audio/` and put the annotations (CSV / Audacity `.txt` / Raven `.txt`) into
> `annotations/`. A dataset organized as one folder per species can be turned
> into a CSV in a few lines of Python (filename → label).


### (Optional) download recordings straight into Drive

If you have direct URLs (e.g. Orcasound / Watkins clips) instead of local
files, fetch them with the `ingest` command. Skip this if you uploaded audio to
Drive manually.

In [ ]:
# Example — replace with your own URLs, or comment out.
# !python whale_bot.py ingest urls \
#     https://example.org/hydrophone/clip1.wav \
#     https://example.org/hydrophone/clip2.wav
#
# Or from a manifest file (one URL per line) placed in Drive:
# !python whale_bot.py ingest manifest {DRIVE_ROOT}/urls.txt

## 5. Process audio + tags → spectrograms

Writes `datasets/spectrograms/{train,val}/<label>/*.png`.

In [ ]:
!python whale_bot.py process --val-fraction 0.2

## 6. Train

The trainer auto-selects the TPU/GPU. On a TPU, prefer a **larger batch size**
(a multiple of 8, e.g. 64) to keep the cores fed and amortize XLA compilation.

In [ ]:
BATCH_SIZE = 64 if USE_TPU else 32
EPOCHS = 15

!python whale_bot.py train --epochs {EPOCHS} --batch-size {BATCH_SIZE} --arch resnet18

## 7. Predict on a recording

In [ ]:
# Point at any recording in data/audio/ (or a whole folder).
import os
audio = sorted(os.listdir("data/audio"))
if audio:
    !python whale_bot.py predict data/audio/{audio[0]}
else:
    print("No audio in data/audio/ — add recordings first.")

## 8. Save the trained model back to Drive

In [ ]:
import shutil
ckpt = "models/whale_classifier.pt"
if os.path.exists(ckpt):
    dest = f"{MODEL_OUTPUT}/whale_classifier.pt"
    shutil.copy2(ckpt, dest)
    print("saved ->", dest)
else:
    print("No checkpoint yet — run training first.")